<a href="https://colab.research.google.com/github/takahashi-crypto556/gemini-app/blob/main/03_business_card_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
!pip install google-genai

In [85]:
import os
from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='こんにちは！開発環境の接続テストです。応答できますか？',
)

print(response.text)


こんにちは！はい、正常に応答しています。接続テストは成功です。

開発環境の準備、お疲れ様です！何かお手伝いできることがあれば、お気軽にお知らせください。


In [86]:
import re

def extract_value(text, field):
    key = str(field).lower()

    patterns = {
        "company_name": r"^\s*(?:company_name|会社名)[:：]\s*(.*)",
        "department":   r"^\s*(?:department|部署名|部署)[:：]\s*(.*)",
        "title":        r"^\s*(?:title|役職)[:：]\s*(.*)",
        "name":         r"^\s*(?:name|name|氏名)[:：]\s*(.*)",
        "name_kana":    r"^\s*(?:name_kana|フリガナ)[:：]\s*(.*)",
        "email":        r"^\s*(?:email|MAIL|メールアドレス)[:：]\s*(.*)",
        "phone":        r"^\s*(?:phone|TEL|電話番号)[:：]\s*(.*)",
        "address":      r"^\s*(?:address|住所)[:：]\s*(.*)",
     }

    pattern = patterns.get(key)
    if pattern:
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        if match and match.group(1):
            val = match.group(1).strip()
            if val:
                return val

    return None

class BusinessCard:
    def __init__(self,text):
        self.Company_name = extract_value(text, "company_name")
        self.Department = extract_value(text, "department")
        self.Title = extract_value(text, "title")
        self.Name = extract_value(text, "name")
        self.Name_kana = extract_value(text, "name_kana")
        self.Email = extract_value(text, "email")
        self.Phone = extract_value(text, "phone")
        self.Address = extract_value(text, "address")

    def __str__(self):
        lines = []
        if self.Company_name:
            lines.append("会社名:" + self.Company_name)
        if self.Department:
            lines.append("部署名:" + self.Department)
        if self.Title:
            lines.append("役職:" + self.Title)
        if self.Name:
            lines.append("氏名:" + self.Name)
        if self.Name_kana:
            lines.append("フリガナ:" + self.Name_kana)
        if self.Email:
            lines.append("メールアドレス:" + self.Email)
        if self.Phone:
            lines.append("電話番号:" + self.Phone)
        if self.Address:
            lines.append("住所:" + self.Address)
        return"\n".join(lines) if lines else "情報なし"

In [87]:
!pip install litellm

In [88]:
import base64
import os
import sys
import litellm
from google.colab import files, userdata

DEFAULT_DIR = "data"
DEFAULT_IMAGE_PATH = os.path.join(DEFAULT_DIR,"sample_card.png")

os.makedirs(DEFAULT_DIR, exist_ok=True)

print("---名刺画像の読み込み---")
print("※ファイルをアップロードするか、キャンセル/スキップして同梱のサンプル画像を使用します。")
uploaded = files.upload()

if uploaded:
    image_path = list(uploaded.keys())[0]
    print(f"アップロードされた画像を使用します:{image_path}")

    with open(DEFAULT_IMAGE_PATH, "wb") as f:
        f.write(uploaded[image_path])
else:
    if os.path.exists(DEFAULT_IMAGE_PATH):
        print(f"同梱のサンプル画像を使用します: {image_path}")
    else:
        raise FileNotFoundError(
            f"エラー: {DEFAULT_IMAGE_PATH}が見つかりません。\n"
            "初回実行時はダイアログから名刺画像をアップロードしてください。"
        )

print("名刺画像を処理中...")
with open(image_path,"rb") as file:
    encoded = base64.b64encode(file.read()).decode('utf-8')

ext = os.path.splitext(image_path)[1].lower()[1:]
if ext not in ('png', 'gif', 'bmp', 'webp'):
    ext = 'jpeg'
image_base64 = f"data:image/{ext};base64,{encoded}"

import litellm
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

prompt_text = (
    "提供された名刺画像からテキスト情報を読み取り指定されたフォーマットのみに従って出力してください\n\n"
    "【出力ルール】\n"
    "-挨拶や解説などの余計な文字列は一切出力せず以下のフォーマットのみを出力してください\n"
    "-項目が存在しないまたは読み取れない場合は「なし」と出力してください\n\n"
    "【出力フォーマット】\n"
    "Company_name: 会社名\n"
    "Department: 部署名\n"
    "Title: 役職\n"
    "Name: 氏名\n"
    "Name_kana: フリガナ\n"
    "Email: メールアドレス\n"
    "Phone: 電話番号\n"
    "Address: 住所\n"
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt_text
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": image_base64
                }
            }
        ]
    }
]

response = litellm.completion(
    model="gemini/gemini-3.6-flash",
    messages=messages
)
ocr_result_text = response.choices[0].message.content

card = BusinessCard(ocr_result_text)
print("抽出されたプロファイル情報:")
print(card)


---名刺画像の読み込み---
※ファイルをアップロードするか、キャンセル/スキップして同梱のサンプル画像を使用します。


Saving Gemini_Generated_Image_deywjjdeywjjdeyw.jpeg to Gemini_Generated_Image_deywjjdeywjjdeyw (5).jpeg
アップロードされた画像を使用します:Gemini_Generated_Image_deywjjdeywjjdeyw (5).jpeg
名刺画像を処理中...
抽出されたプロファイル情報:
会社名:株式会社ソリューション・ブリッジ
部署名:システム開発部
役職:代表取締役
氏名:佐藤 健太
フリガナ:サトウ ケンタ
メールアドレス:k.sato@solubridge.co.jp
電話番号:052-123-4567
住所:〒460-0008 愛知県名古屋市中区栄1-2-3


In [89]:
#テスト用のダミー名刺テキスト
sample_text ="""
Company_name:株式会社テクノロジーラボ
Title:技術開発部 ディレクター
Name:山田 太郎

Address:〒100-0001 東京都千代田区1-1-1
TEL:03-1234-5678
MAIL:yamada@example.com
"""
card = BusinessCard(sample_text)

print("氏名:", card.Name)
print("会社名:", card.Company_name)

氏名: 山田 太郎
会社名: 株式会社テクノロジーラボ
